# Scoping Review Screener — AI in Preterm Birth Prediction

**Model:** Qwen3:8b via Ollama (free, runs locally)  
**Input:** `rayyan_resolved_duplicates.csv` (9,504 records, 77.9% with abstracts)  

**Before running:**
- Make sure Ollama is running: open Terminal and run `ollama serve`
- Verify model is available: `ollama list` should show `qwen3:8b`

**Pipeline:**
```
rayyan_resolved_duplicates.csv
     ↓
Parse + clean each record
     ↓
Send title + abstract + keywords to Qwen3:8b via Ollama
     ↓
JSON: include / exclude / uncertain
     ↓
Checkpoint every 50 articles
     ↓
Export Excel + CSV
```

In [ ]:
# Install dependencies 
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'requests', 'pandas', 'openpyxl', 'tqdm'])

In [ ]:
# Configuration
import re, json, time, html
import requests
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import os

# Ollama settings
OLLAMA_URL    = "http://localhost:11434/api/chat"
MODEL         = "qwen3:8b"

# Input file
INPUT_CSV     = "./Metadata/rayyan_resolved_duplicates.csv"

# Output files 
os.makedirs("./Screening", exist_ok=True)
OUTPUT_EXCEL     = "./Screening/screening_results_preterm.xlsx"
OUTPUT_CSV       = "./Screening/screening_results_preterm.csv"
CHECKPOINT_CSV   = "./Screening/screening_checkpoint_preterm.csv"
CHECKPOINT_EXCEL = "./Screening/screening_checkpoint_preterm.xlsx"

# Performance settings
CHECKPOINT_EVERY = 50
TIMEOUT_SECONDS  = 120  

print("Configuration loaded.")
print(f"   Model     : {MODEL}")
print(f"   Ollama URL: {OLLAMA_URL}")
print(f"   Input file: {INPUT_CSV}")

In [ ]:
# Check Ollama is running
import requests

try:
    resp = requests.get("http://localhost:11434", timeout=5)
    print("Ollama is running!")
except Exception as e:
    print("Ollama is NOT running.")
    print("Open Terminal and run: ollama serve")
    print(f"Error: {e}")

In [ ]:
# Screening Criteria

INCLUSION_CRITERIA = """
POPULATION:
- Pregnant women, regardless of age or type of pregnancy (singleton or multiple)
- Human subjects only

INTERVENTION / TECHNOLOGY:
- Use of artificial intelligence (AI), machine learning (ML), deep learning,
  neural networks, natural language processing, or any AI-based algorithm

OUTCOME:
- Study aims to identify, predict, detect, or screen for the risk of preterm birth
- Preterm birth = birth before 37 weeks of gestation
- Includes: spontaneous or induced preterm, very preterm (<32wk), extremely preterm (<28wk)

CONTEXT:
- Prenatal care setting (clinical, hospital, community, or population-based)
- Uses real clinical data or population databases

STUDY DESIGN:
- Quantitative observational studies (prospective or retrospective)
- AI model development or validation studies with clinical application
- Published in French or English
"""

EXCLUSION_CRITERIA = """
- Animal or purely experimental (non-human) studies
- Purely technical/computational papers with no explicit clinical application to preterm birth
- Editorial, commentary, letter, opinion, or narrative without original data
- Study protocol without results
- Conference abstract without full text
- Systematic reviews, scoping reviews, meta-analyses, literature reviews
- Studies focused ONLY on postpartum/neonatal/postnatal outcomes (no prenatal component)
- Studies unrelated to preterm birth prediction (e.g. AI for GDM, preeclampsia,
  fetal growth — unless preterm birth is also a stated outcome)
- Studies without any AI/ML component
"""

print("Screening criteria loaded.")

In [ ]:
# Load and clean csv

def clean(value):
    if pd.isna(value) or value is None:
        return ""
    s = str(value).strip()
    s = html.unescape(s)
    s = re.sub(r'\s+', ' ', s)
    s = re.sub(r'^Abstract\s*[-–]\s*', '', s, flags=re.IGNORECASE)
    return s.strip()


def load_articles(filepath):
    if not Path(filepath).exists():
        raise FileNotFoundError(f"File not found: {filepath}")

    df = pd.read_csv(filepath, dtype=str, low_memory=False)
    print(f"Raw rows          : {len(df)}")

    articles = []
    seen = set()

    for _, row in df.iterrows():
        title    = clean(row.get('title', ''))
        abstract = clean(row.get('abstract', ''))

        if not title and not abstract:
            continue

        doi = clean(row.get('doi', ''))
        dedup_key = doi.lower() if doi else title.lower()[:100]
        if dedup_key in seen:
            continue
        seen.add(dedup_key)

        articles.append({
            'rayyan_key' : clean(row.get('key', '')),
            'title'      : title,
            'authors'    : clean(row.get('authors', '')),
            'year'       : clean(row.get('year', '')),
            'journal'    : clean(row.get('journal', '')),
            'abstract'   : abstract,
            'keywords'   : clean(row.get('keywords', ''))[:400],
            'doi'        : doi,
            'pubmed_id'  : clean(row.get('pubmed_id', '')),
            'url'        : clean(row.get('url', '')),
            'language'   : clean(row.get('language', '')),
        })

    has_abstract = sum(1 for a in articles if a['abstract'])
    print(f"Unique articles   : {len(articles)}")
    print(f"With abstract     : {has_abstract} ({has_abstract/len(articles)*100:.1f}%)")
    print(f"Without abstract  : {len(articles)-has_abstract} ({(len(articles)-has_abstract)/len(articles)*100:.1f}%)")
    return articles


print("Loading articles...")
all_articles = load_articles(INPUT_CSV)
print(f"\nReady to screen {len(all_articles)} articles.")

In [ ]:
# Qwen3 Screener via Ollama

def build_prompt(article):
    abstract_text = article['abstract'][:1500] if article['abstract'] \
                else 'Not available.'
    return f"""/no_think
You are screening articles for a scoping review. Respond with JSON only, nothing else.

Does this article use AI/ML to PREDICT or IDENTIFY PRETERM BIRTH RISK in PREGNANT WOMEN during PRENATAL CARE?
Preterm birth = birth before 37 weeks gestation.

Exclude if: animal study, postpartum only, unrelated disease, no AI, review article, protocol, conference abstract.

Return only this JSON, no other text:
{{"decision":"include" or "exclude" or "uncertain","confidence":"high" or "medium" or "low","main_reason":"one sentence","study_design":"unclear","ai_method_detected":"unclear","preterm_birth_mentioned":true or false}}

Title: {article['title']}
Abstract: {abstract_text}
"""


def extract_json(text):
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()
    start = text.find('{')
    if start == -1:
        raise ValueError(f"No JSON found: {text[:200]}")
    text = text[start:]
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    result = {
        'decision': 'uncertain', 'confidence': 'low',
        'main_reason': '', 'matched_inclusion_criteria': [],
        'matched_exclusion_criteria': [], 'study_design': 'unclear',
        'ai_method_detected': 'unclear', 'preterm_birth_mentioned': False
    }
    for field, pattern in [
        ('decision',           r'"decision"\s*:\s*"(include|exclude|uncertain)"'),
        ('confidence',         r'"confidence"\s*:\s*"(high|medium|low)"'),
        ('main_reason',        r'"main_reason"\s*:\s*"([^"]+)"'),
        ('study_design',       r'"study_design"\s*:\s*"([^"]+)"'),
        ('ai_method_detected', r'"ai_method_detected"\s*:\s*"([^"]+)"'),
    ]:
        m = re.search(pattern, text, re.IGNORECASE)
        if m:
            result[field] = m.group(1)
    ptb = re.search(r'"preterm_birth_mentioned"\s*:\s*(true|false)', text, re.IGNORECASE)
    if ptb:
        result['preterm_birth_mentioned'] = ptb.group(1).lower() == 'true'
    return result


def screen_article(article, retries=1):
    prompt = build_prompt(article)
    for attempt in range(retries):
        try:
            response = requests.post(
                "http://localhost:11434/api/chat",
                json={
                    "model": MODEL,
                    "messages": [{"role": "user", "content": prompt}],
                    "stream": False,
                    "options": {
                        "temperature": 0,
                        "num_predict": 800
                    }
                },
                timeout=TIMEOUT_SECONDS
            )
            response.raise_for_status()
            raw = response.json()["message"]["content"].strip()
            if not raw:
                raise ValueError("Empty response")
            return extract_json(raw)
        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")
            if attempt < retries - 1:
                time.sleep(2)

    title_lower = article['title'].lower()
    if any(w in title_lower for w in ['preterm', 'premature birth',
                                       'premature labor', 'premature labour',
                                       'preterm labor']):
        return {'decision': 'uncertain', 'confidence': 'low',
                'main_reason': 'Preterm in title — needs manual review',
                'matched_inclusion_criteria': [], 'matched_exclusion_criteria': [],
                'study_design': 'unclear', 'ai_method_detected': 'unclear',
                'preterm_birth_mentioned': True}
    return {'decision': 'uncertain', 'confidence': 'low',
            'main_reason': 'Model failed — needs manual review',
            'matched_inclusion_criteria': [], 'matched_exclusion_criteria': [],
            'study_design': 'unclear', 'ai_method_detected': 'unclear',
            'preterm_birth_mentioned': False}

print("Screener ready.")

In [ ]:
# Test on 5 articles

print("Testing Qwen3 on 5 articles...\n")

with_abs    = [a for a in all_articles if a['abstract']][:3]
without_abs = [a for a in all_articles if not a['abstract']][:2]

for i, article in enumerate(with_abs + without_abs, 1):
    tag = "Abstract" if article['abstract'] else "Text"
    print(f"{i}. {tag} {article['title'][:65]}")
    result = screen_article(article)
    print(f" - {result['decision'].upper()} ({result['confidence']})")
    print(f" - {result['main_reason']}\n")

print("Test done. If results look correct, proceed to Cell 8.")

In [ ]:
# CELL 8 — Resume from checkpoint

results        = []
processed_keys = set()

if Path(CHECKPOINT_CSV).exists():
    existing_df    = pd.read_csv(CHECKPOINT_CSV)
    results        = existing_df.to_dict('records')
    processed_keys = set(
        str(r.get('doi', '')).lower() + '||' +
        str(r.get('title', '')).lower()[:80]
        for r in results
    )
    print(f"Resuming: {len(results)} done, {len(all_articles) - len(results)} remaining.")
else:
    print(f"No checkpoint. Starting from zero. {len(all_articles)} articles to screen.")

In [ ]:
# Full Screening Run
# Run caffeinate -i & in Terminal first to prevent sleep! (Mac)
# Checkpoints every 50 articles (safe to stop and resume).

def save_checkpoint(records):
    pd.DataFrame(records).to_csv(CHECKPOINT_CSV, index=False)
    pd.DataFrame(records).to_excel(CHECKPOINT_EXCEL, index=False)


start   = time.perf_counter()
skipped = 0

for i, article in enumerate(tqdm(all_articles, desc="Screening"), start=1):
    key = (
        str(article.get('doi', '')).lower() + '||' +
        str(article.get('title', '')).lower()[:80]
    )

    if key in processed_keys:
        skipped += 1
        continue

    decision = screen_article(article)

    results.append({
        'rayyan_key'             : article['rayyan_key'],
        'title'                  : article['title'],
        'authors'                : article['authors'],
        'year'                   : article['year'],
        'journal'                : article['journal'],
        'doi'                    : article['doi'],
        'pubmed_id'              : article['pubmed_id'],
        'url'                    : article['url'],
        'language'               : article['language'],
        'keywords'               : article['keywords'][:300],
        'abstract_available'     : bool(article['abstract']),
        'decision'               : decision.get('decision', 'uncertain'),
        'confidence'             : decision.get('confidence', 'low'),
        'main_reason'            : decision.get('main_reason', ''),
        'matched_inclusion'      : '; '.join(
                                     decision.get('matched_inclusion_criteria', [])),
        'matched_exclusion'      : '; '.join(
                                     decision.get('matched_exclusion_criteria', [])),
        'study_design'           : decision.get('study_design', ''),
        'ai_method_detected'     : decision.get('ai_method_detected', ''),
        'preterm_birth_mentioned': decision.get('preterm_birth_mentioned', False),
    })

    processed_keys.add(key)

    if i % CHECKPOINT_EVERY == 0:
        save_checkpoint(results)
        inc = sum(1 for r in results if r['decision'] == 'include')
        elapsed_h = (time.perf_counter() - start) / 3600
        rate = (i - skipped) / elapsed_h if elapsed_h > 0 else 0
        remaining = len(all_articles) - i
        eta_h = remaining / rate if rate > 0 else 0
        print(f"\n{i}/{len(all_articles)} | Included: {inc} | "
              f"Speed: {rate:.0f}/hr | ETA: {eta_h:.1f}h")

# Final save
df_final = pd.DataFrame(results)
df_final.to_excel(OUTPUT_EXCEL, index=False)
df_final.to_csv(OUTPUT_CSV, index=False)

elapsed = time.perf_counter() - start
print("\n" + "="*50)
print("Screening complete!")
print(f"Processed : {len(results)}")
print(f"Skipped   : {skipped} (checkpoint)")
print(f"Time      : {elapsed/3600:.2f} hours")

In [ ]:
# Summary Statistics

df = pd.read_csv(OUTPUT_CSV)
total     = len(df)
included  = (df['decision'] == 'include').sum()
excluded  = (df['decision'] == 'exclude').sum()
uncertain = (df['decision'] == 'uncertain').sum()
no_abs    = (~df['abstract_available'].astype(bool)).sum()
errors    = df['main_reason'].str.contains('error', case=False, na=False).sum()

print("="*50)
print("SCREENING RESULTS SUMMARY")
print("="*50)
print(f" - Total screened : {total}")
print(f" - Included       : {included} ({included/total*100:.1f}%)")
print(f" - Excluded       : {excluded} ({excluded/total*100:.1f}%)")
print(f" - Uncertain      : {uncertain} ({uncertain/total*100:.1f}%)")
print(f" - No abstract    : {no_abs} ({no_abs/total*100:.1f}%)")
print(f" - Errors         : {errors}\n")

inc_df = df[df['decision'] == 'include']
if len(inc_df) > 0:
    print("\nConfidence (included):")
    print(inc_df['confidence'].value_counts().to_string())
    print("\nTop AI methods (included):")
    print(inc_df['ai_method_detected'].value_counts().head(15).to_string())
    print("\nStudy designs (included):")
    print(inc_df['study_design'].value_counts().head(10).to_string())
    print("\nYear distribution (included):")
    print(inc_df['year'].value_counts().sort_index().to_string())

In [ ]:
# Export separated results

df = pd.read_csv(OUTPUT_CSV)

included_df  = df[df['decision'] == 'include'].copy()
uncertain_df = df[df['decision'] == 'uncertain'].copy()
excluded_df  = df[df['decision'] == 'exclude'].copy()

error_df    = uncertain_df[
    uncertain_df['main_reason'].str.contains('error', case=False, na=False)].copy()
genuine_unc = uncertain_df[
    ~uncertain_df['main_reason'].str.contains('error', case=False, na=False)].copy()

included_df.to_excel('included_articles_preterm.xlsx',  index=False)
included_df.to_csv('included_articles_preterm.csv',     index=False)
genuine_unc.to_excel('uncertain_articles_preterm.xlsx', index=False)
genuine_unc.to_csv('uncertain_articles_preterm.csv',    index=False)
error_df.to_excel('errors_preterm.xlsx',                index=False)
excluded_df.to_excel('excluded_articles_preterm.xlsx',  index=False)

print(f"{len(included_df):>5} included - included_articles_preterm.xlsx")
print(f"{len(genuine_unc):>5} uncertain - uncertain_articles_preterm.xlsx (review manually)")
print(f"{len(error_df):>5} errors - errors_preterm.xlsx (re-run these)")
print(f"{len(excluded_df):>5} excluded - excluded_articles_preterm.xlsx")

print("\nNext steps:")
print("1. Manually review uncertain articles")
print("2. Re-run notebook for any error articles")
print("3. Update PRISMA flow diagram with final numbers")